In [1]:
# 1. Importação de pacotes
import pandas as pd
from sklearn.impute import SimpleImputer

# 2. Carregar a base de dados diretamente da nuvem
url = 'https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv'
df_gripe = pd.read_csv(url)

# 3. Limpeza do ficheiro e renomeação de colunas
df_gripe = df_gripe.drop(columns=['Carimbo de data/hora'])
df_gripe.columns = ['ficou_gripado', 'vacina', 'aglomeracao', 'viajou', 'alergia', 'sono', 'exercicio', 'alimentacao', 'lavou_maos', 'estresse']
df_gripe['estresse'] = df_gripe['estresse'].astype(str)

# 4. Separação entre as características (X) e a classe alvo (y)
X_bruto = df_gripe.drop(columns=['ficou_gripado'])
y_alvo = df_gripe['ficou_gripado']

# 5. Tratamento de valores nulos (Missing Values)
imputador = SimpleImputer(strategy='most_frequent')
X_limpo = pd.DataFrame(imputador.fit_transform(X_bruto), columns=X_bruto.columns)

print("Ficheiro carregado e dados nulos tratados com sucesso!")

Ficheiro carregado e dados nulos tratados com sucesso!


In [2]:
# Transformação para binário
# Exemplo: A coluna 'alergia' torna-se 'alergia_Sim' (1) e 'alergia_Não' (0)
X_regras = pd.get_dummies(X_limpo)

print("Atributos mapeados para lógica booleana (ideal para regras de cobertura):")
display(X_regras.head(3))

Atributos mapeados para lógica booleana (ideal para regras de cobertura):


,vacina_Não,vacina_Sim,aglomeracao_Não,aglomeracao_Sim,viajou_Nuca,viajou_Pelo menos uma vez por mês,viajou_Poucas vezes por ano,alergia_Muito,alergia_Médio,alergia_Não,...,lavou_maos_2 vezes ou menos,lavou_maos_3 a 5 vezes,lavou_maos_6 a 10 vezes,lavou_maos_Mais de 10 vezes,estresse_1.0,estresse_2.0,estresse_3.0,estresse_4.0,estresse_5.0,estresse_nan
0,False,True,False,True,False,False,True,False,True,False,...,False,True,False,False,False,False,False,False,True,False
1,False,True,False,True,True,False,False,False,False,True,...,False,False,False,True,False,False,True,False,False,False
2,True,False,False,True,False,False,True,False,False,False,...,False,False,True,False,False,False,True,False,False,False


In [3]:
from sklearn.tree import DecisionTreeClassifier

# Instanciar o indutor (max_depth=3 garante que as regras não ficam complexas demais)
indutor_regras = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)

# O processo de treino para encontrar a melhor separação de atributos
indutor_regras.fit(X_regras, y_alvo)

print("Indução concluída! As regras foram criadas com base no ganho de informação (entropia).")

Indução concluída! As regras foram criadas com base no ganho de informação (entropia).


In [4]:
from sklearn.tree import export_text

# Extração das regras num formato de texto legível por humanos
regras_texto = export_text(indutor_regras, feature_names=list(X_regras.columns))

print("=== CONJUNTO DE REGRAS DE CLASSIFICAÇÃO ===")
print("Lógica extraída pelo modelo (If -> Then):\n")
print(regras_texto)

=== CONJUNTO DE REGRAS DE CLASSIFICAÇÃO ===
Lógica extraída pelo modelo (If -> Then):

|--- viajou_Pelo menos uma vez por mês <= 0.50
|   |--- estresse_5.0 <= 0.50
|   |   |--- vacina_Não <= 0.50
|   |   |   |--- class: Não
|   |   |--- vacina_Não >  0.50
|   |   |   |--- class: Sim
|   |--- estresse_5.0 >  0.50
|   |   |--- alergia_Médio <= 0.50
|   |   |   |--- class: Sim
|   |   |--- alergia_Médio >  0.50
|   |   |   |--- class: Sim
|--- viajou_Pelo menos uma vez por mês >  0.50
|   |--- alergia_Médio <= 0.50
|   |   |--- aglomeracao_Sim <= 0.50
|   |   |   |--- class: Não
|   |   |--- aglomeracao_Sim >  0.50
|   |   |   |--- class: Não
|   |--- alergia_Médio >  0.50
|   |   |--- estresse_2.0 <= 0.50
|   |   |   |--- class: Sim
|   |   |--- estresse_2.0 >  0.50
|   |   |   |--- class: Não



In [5]:
# 1. Simulação de um novo paciente no consultório
novo_caso = pd.DataFrame([{
    'vacina': 'Sim',
    'aglomeracao': 'Sim',
    'viajou': 'Poucas vezes por ano',
    'alergia': 'Não',
    'sono': 'mais de 6 horas',
    'exercicio': 'Sim',
    'alimentacao': 'Às vezes',
    'lavou_maos': 'Mais de 10 vezes',
    'estresse': '2'
}])

# 2. Alinhamento de atributos (garante que as regras testam as colunas corretas)
caso_transformado = pd.get_dummies(novo_caso)
caso_alvo = caso_transformado.reindex(columns=X_regras.columns, fill_value=0)

# 3. Classificação baseada no conjunto de regras gerado
previsao_regra = indutor_regras.predict(caso_alvo)[0]

print("="*40)
print(" 📜 DIAGNÓSTICO POR REGRAS DE COBERTURA")
print("="*40)
print(f"Resultado da Classificação: O paciente FICOU GRIPADO? -> {previsao_regra.upper()}")

 📜 DIAGNÓSTICO POR REGRAS DE COBERTURA
Resultado da Classificação: O paciente FICOU GRIPADO? -> NÃO
